MongoDB connection

In [1]:
from pymongo import MongoClient

In [2]:
client = MongoClient("mongodb://localhost:27017/")

In [3]:
db = client["ecommerce_recommendation"]

In [4]:
reviews_collection = db["reviews"]
interactions_collection = db["interactions"]

JOB 1

Counting How Many Times each User Action occur

In [ ]:
map_function = """
function() {
    emit(this.action, 1);
}
"""

In [6]:
reduce_function = """
function(key, values) {
    return Array.sum(values);
}
"""

In [7]:
result = db.command({
    "mapReduce": "interactions",

    "map": map_function,

    "reduce": reduce_function,

    "out": "action_frequency"
})

In [8]:
list(db["action_frequency"].find())

[{'_id': 'ignore', 'value': 4009.0},
 {'_id': 'purchase', 'value': 24749.0},
 {'_id': 'view', 'value': 3178.0},
 {'_id': 'add_to_cart', 'value': 8063.0}]

Equivalent aggregation pipeline

In [9]:
pipeline = [
    {
        "$group": {

            "_id": "$action",

            "count": {
                "$sum": 1
            }
        }
    }
]

In [10]:
result = interactions_collection.aggregate(pipeline)

In [13]:
for item in result:
    print(item)

{'_id': 'ignore', 'count': 4009}
{'_id': 'purchase', 'count': 24749}
{'_id': 'view', 'count': 3178}
{'_id': 'add_to_cart', 'count': 8063}


JOB 2

Calculating total rating score per year

In [14]:
map_function = """
function() {
    emit(this.year, this.overall);
}
"""

In [15]:
reduce_function = """
function(key, values) {
    return Array.sum(values);
}
"""

In [16]:
result = db.command({
    "mapReduce": "reviews",

    "map": map_function,

    "reduce": reduce_function,

    "out": "rating_per_year"
})

In [17]:
list(db["rating_per_year"].find())

[{'_id': 2005.0, 'value': 3960.0},
 {'_id': 2008.0, 'value': 8720.0},
 {'_id': 2010.0, 'value': 10636.0},
 {'_id': 2013.0, 'value': 41823.0},
 {'_id': 2011.0, 'value': 13854.0},
 {'_id': 2012.0, 'value': 19001.0},
 {'_id': 2009.0, 'value': 9160.0},
 {'_id': 2002.0, 'value': 8863.0},
 {'_id': 2003.0, 'value': 6274.0},
 {'_id': 2004.0, 'value': 3718.0},
 {'_id': 2000.0, 'value': 3531.0},
 {'_id': 1999.0, 'value': 316.0},
 {'_id': 2014.0, 'value': 20667.0},
 {'_id': 2007.0, 'value': 8920.0},
 {'_id': 2006.0, 'value': 4783.0},
 {'_id': 2001.0, 'value': 6738.0}]

Equivalent aggregation pipeline

In [18]:
pipeline = [
    {
        "$group": {

            "_id": "$year",

            "total_rating": {
                "$sum": "$overall"
            }
        }
    }
]

In [19]:
result = reviews_collection.aggregate(pipeline)

In [20]:
for item in result:
    print(item)

{'_id': 2013, 'total_rating': 41823}
{'_id': 2011, 'total_rating': 13854}
{'_id': 2001, 'total_rating': 6738}
{'_id': 2008, 'total_rating': 8720}
{'_id': 2005, 'total_rating': 3960}
{'_id': 2007, 'total_rating': 8920}
{'_id': 2004, 'total_rating': 3718}
{'_id': 2002, 'total_rating': 8863}
{'_id': 1999, 'total_rating': 316}
{'_id': 2014, 'total_rating': 20667}
{'_id': 2010, 'total_rating': 10636}
{'_id': 2012, 'total_rating': 19001}
{'_id': 2009, 'total_rating': 9160}
{'_id': 2006, 'total_rating': 4783}
{'_id': 2003, 'total_rating': 6274}
{'_id': 2000, 'total_rating': 3531}


Which products generate the strongest customer engagement?

engagement = helpful_ratio × review_length

In [21]:
map_function = """
function() {

    var engagement =
        this.analytics.helpful_ratio *
        this.analytics.review_length;

    emit(this.asin, engagement);
}
"""

In [22]:
reduce_function = """
function(key, values) {
    return Array.sum(values);
}
"""

In [23]:
result = db.command({
    "mapReduce": "reviews",

    "map": map_function,

    "reduce": reduce_function,

    "out": "product_engagement"
})

In [24]:
list(db["product_engagement"].find().limit(10))

[{'_id': 'B00005JJEO', 'value': 1098.8333333398},
 {'_id': 'B00004T1XE', 'value': 4367.2810749604005},
 {'_id': 'B000063UZW', 'value': 1089.7454545458},
 {'_id': 'B000067RJB', 'value': 340.0},
 {'_id': 'B00004TX77', 'value': 397.5},
 {'_id': 'B000050XL7', 'value': 4308.5047442484},
 {'_id': 'B00005NIMJ', 'value': 12700.9544675445},
 {'_id': 'B00004YV85', 'value': 1250.4025210070001},
 {'_id': 'B00005N5WU', 'value': 2021.3043478242},
 {'_id': 'B00006B7RS', 'value': 0.0}]

Equivalent aggregation pipeline

In [25]:
pipeline = [
    {
        "$project": {

            "asin": 1,

            "engagement_score": {

                "$multiply": [
                    "$analytics.helpful_ratio",
                    "$analytics.review_length"
                ]
            }
        }
    },
    {
        "$group": {

            "_id": "$asin",

            "total_engagement": {
                "$sum": "$engagement_score"
            }
        }
    }
]

In [26]:
result = reviews_collection.aggregate(pipeline)

In [27]:
for item in result:
    print(item)

{'_id': 'B00006B9W3', 'total_engagement': 1284.0000000088}
{'_id': 'B000056ULH', 'total_engagement': 2226.1604617591}
{'_id': 'B00004Z6R4', 'total_engagement': 283.0}
{'_id': 'B000068ILF', 'total_engagement': 168.0}
{'_id': 'B00003OPEV', 'total_engagement': 1243.0933002305}
{'_id': 'B00004UDQ1', 'total_engagement': 1581.3788820061}
{'_id': 'B00004WHFK', 'total_engagement': 2522.0109227907}
{'_id': 'B00005T408', 'total_engagement': 2416.1652913843}
{'_id': 'B000068IGQ', 'total_engagement': 560.0}
{'_id': 'B00000DM9W', 'total_engagement': 2750.9999999898}
{'_id': 'B00004Z8SC', 'total_engagement': 607.0}
{'_id': 'B00005MNSS', 'total_engagement': 2391.6090909038003}
{'_id': 'B000069K8N', 'total_engagement': 4573.4450793456}
{'_id': 'B00005N6KF', 'total_engagement': 1648.2878788024}
{'_id': 'B000067RJB', 'total_engagement': 340.0}
{'_id': '8918010656', 'total_engagement': 639.0}
{'_id': 'B00001P4ZH', 'total_engagement': 62271.5435383425}
{'_id': 'B00006B92A', 'total_engagement': 68.3}
{'_id